##### Task 6
Validate data types and formats.

What I want in the end:
- A table of orders with invalid order_date
- A table of items with invalid or missing price
- A table of order lines with invalid quantity
- Counts for each validation failure type

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
-- A table of orders with invalid order_date

SELECT *
FROM bronze_orders
WHERE try_to_date(order_date, 'yyyy-MM-dd') IS NULL;

In [0]:
%sql
-- A table of items with invalid or missing price

SELECT * 
FROM bronze_items
WHERE price IS NULL;

In [0]:
%sql
-- A table of order lines with invalid quantity

CREATE OR REPLACE TEMP VIEW orders_items_flatten AS
SELECT order_id, customer_id, item.item_id item_id, item.quantity quantity, order_date, status
FROM bronze_orders
LATERAL VIEW explode(items) AS item;

SELECT *
FROM orders_items_flatten
WHERE quantity IS NULL OR quantity <= 0;

In [0]:
%sql
-- Counts for each validation failure type

CREATE OR REPLACE TEMP VIEW orders_items_flatten AS
SELECT order_id, customer_id, item.item_id item_id, item.quantity quantity, order_date, status
FROM bronze_orders
LATERAL VIEW explode(items) AS item;

SELECT *
FROM (
  SELECT 'invalid_order_date' validation_failure_type, COUNT(*) failure_count
    FROM bronze_orders
    WHERE try_to_date(order_date, 'yyyy-MM-dd') IS NULL
  UNION ALL
  SELECT 'invalid_or_missing_price', COUNT(*)
    FROM bronze_items
    WHERE price IS NULL
  UNION ALL
  SELECT 'invalid_quantity', COUNT(*)
    FROM orders_items_flatten
    WHERE quantity IS NULL OR quantity <= 0
) t